In [ ]:
# IMPORTS
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

In [ ]:
# DATA LOADING
def load_data(path: str) -> pd.DataFrame:
  return pd.read_csv(path)

In [ ]:
# BASIC PREPROCESS (non-ML transforms)
def preprocess(df: pd.DataFrame) -> pd.DataFrame:
  if "LoanID" in df.columns:
    df = df.drop(columns=["LoanID"])

  return df

In [ ]:
# TRAIN / TEST SPLIT
def split_data(df, target):
  X = df.drop(columns=[target])
  y = df[target]

  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

  return X_train, X_test, y_train, y_test

In [ ]:
# BUILD MODEL PIPELINE (Preprocessing + SMOTE + RF)
def build_model(X_train):

  categorical_cols =X_train.select_dtypes(include=["object"]).columns.tolist()
  numerical_cols = X_train.select_dtypes(exclude=["object"]).columns.tolist()

  # Numeric Pipeline: impute + scale
  numeric_transformer = Pipeline(
      steps=[
          ("imputer", SimpleImputer(strategy="median")),
          ("scaler", StandardScaler())
      ]
  )

  # Categorical pipeline: impute + onehotencoder
  categorical_transformer = Pipeline(
      steps=[
          ("imputer", SimpleImputer(strategy="most_frequent")),
          ("encoder", OneHotEncoder(handle_unknown="ignore"))
      ]
  )

  # Column wise preprocessing
  preprocessor = ColumnTransformer(
      transformers=[
          ("num", numeric_transformer, numerical_cols),
          ("cat", categorical_transformer, categorical_cols)
      ]
  )

  model = ImbPipeline(
      steps=[
          ("preprocessor", preprocessor),
          ("smote", SMOTE(random_state=42)),
          ("classifier", XGBClassifier(
              objective = "binary:logistic",
              eval_metrics = "logloss",
              tree_method = "hist",
              random_state=42,
              n_jobs=1
          )),
      ]
  )

  return model

In [ ]:
# TRAIN + EVALUATE
from sklearn.model_selection import RandomizedSearchCV
def train_and_evaluate(model, X_train, X_test, y_train, y_test):

  param_dist = {
      "classifier__n_estimators": [200,300,400,500],
      "classifier__learning_rate": [0.01,0.05,0.1,0.2],
      "classifier__max_depth": [3,5,7,10],
      "classifier__subsample": [0.6,0.8,1.0],
      "classifier__colsample_bytree": [0.6,0.8,1.0],
      "classifier__gamma": [0,1,2],
      "classifier__reg_lambda": [0.1,1,5,10]

  }

  tuner = RandomizedSearchCV(
      estimator=model,
      param_distributions=param_dist,
      n_iter=20,
      scoring="roc_auc",
      cv=3,
      verbose=1,
      random_state=42,
      n_jobs=-1
  )








  print("Fitting the model.....")
  tuner.fit(X_train,y_train)

  print("\nBest XGB Params: ", tuner.best_params_)
  print("\n Best CV ROC-AUC: ", tuner.best_score_)

  y_preds = tuner.best_estimator_.predict(X_test)
  y_pred_prob = tuner.best_estimator_.predict_proba(X_test)[:,1]

  print("Classification Report:\n")
  print(classification_report(y_test, y_preds))

  roc = roc_auc_score(y_test, y_pred_prob)
  print(f"ROC-AUC: {roc:.3f}")

  print("Accuracy Score:",accuracy_score(y_test, y_preds))

  return tuner.best_estimator_, model

In [ ]:
# Main
def main():

  df = load_data("/content/loan_data.csv")

  df = preprocess(df)

  X_train, X_test, y_train, y_test = split_data(df, target="Default")

  model = build_model(X_train)

  train = train_and_evaluate(model, X_train, X_test, y_train, y_test)

if __name__ == "__main__":
  main()

Fitting the model.....
Fitting 3 folds for each of 20 candidates, totalling 60 fits


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [19:10:19] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "eval_metrics" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Best XGB Params:  {'classifier__subsample': 0.8, 'classifier__reg_lambda': 10, 'classifier__n_estimators': 200, 'classifier__max_depth': 3, 'classifier__learning_rate': 0.2, 'classifier__gamma': 2, 'classifier__colsample_bytree': 0.8}

 Best CV ROC-AUC:  0.7481161107469148
Classification Report:

              precision    recall  f1-score   support

           0       0.89      0.99      0.94     45170
           1       0.57      0.09      0.16      5900

    accuracy                           0.89     51070
   macro avg       0.73      0.54      0.55     51070
weighted avg       0.86      0.89      0.85     51070

ROC-AUC: 0.754
Accuracy Score: 0.8871940473859409
